In [1]:
import os
from dotenv import load_dotenv
# load all environment variables
load_dotenv()
# Load OpenAI API key into environment variable
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# Load groq API key into environment variable
os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")
# LangSmith Tracking configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY_LangTrans"] = os.getenv("LANGCHAIN_API_KEY_LangTrans")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROK_API_KEY")

In [5]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000190B3061110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000190B3A761D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage
## Construct messages that will be sent to the model
messages = [
    SystemMessage(content="Translate the following from English to Arabic."),
    HumanMessage(content="Hello, how are you?")
]

result_ai_message = llm.invoke(messages)

In [9]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
llm_response = parser.invoke(result_ai_message)
llm_response

'مرحبا، كيف حالك؟'

In [11]:
## Using LCEL, we can chain the components together (llm model and parser)
chain = llm | parser
final_response = chain.invoke(messages)
final_response

'مرحبا، كيف حالك؟ \n\nTranslation:\n\n- مرحبا (Merhaba) = Hello\n- كيف حالك (Kayf haaluk) = How are you'

In [10]:
## Prompt template approach
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
generic_template = "Translate the following into {language}."
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(generic_template),
    HumanMessagePromptTemplate.from_template("{text_to_translate}")
])

In [13]:
result = prompt.invoke({"language": "Arabic", "text_to_translate": "Good morning, have a nice day!"})
result.to_messages()

[SystemMessage(content='Translate the following into Arabic.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Good morning, have a nice day!', additional_kwargs={}, response_metadata={})]

In [14]:
chain = prompt | llm | parser
final_response = chain.invoke({"language": "Arabic", "text_to_translate": "Good morning, have a nice day!"})
final_response

'السلام عليكم, أتمنى لكم يومًا ممتعًا.'